# 🇲🇦 AEGIS Darija Dataset Preparation

**Purpose**: Merge the two OMCD datasets (Mendeley V4 + GitHub kabinlessefar) into one unified, deduplicated training set for fine-tuning DarijaBERT-mix.

**Datasets**:
1. **Mendeley V4** — ~20,402 sentences, relabeled by Claude 3.5 Sonnet (from your Drive: `MyDrive/AEGIS_ML/data/darija/Moroccan Darija Offensive Language Detection Datas.zip`)
2. **GitHub OMCD** — `train.csv` + `test.csv` from kabilessefar (from your Drive: `MyDrive/AEGIS_ML/data/darija/Offensive_Moroccan_Comments_datas/`)

**Output**: Cleaned, merged CSV files ready for DarijaBERT-mix fine-tuning.

## 1. Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import zipfile
import io

# Base paths on Google Drive
DRIVE_BASE = '/content/drive/MyDrive/AEGIS_ML/data/darija'
MENDELEY_ZIP = os.path.join(DRIVE_BASE, 'Moroccan Darija Offensive Language Detection Datas.zip')
GITHUB_DIR = os.path.join(DRIVE_BASE, 'Offensive_Moroccan_Comments_datas')

# Output directory (also on Drive so it persists)
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'merged_dataset')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'✅ Output dir: {OUTPUT_DIR}')

## 2. Explore Dataset 1: Mendeley V4

Unzip and examine the structure.

In [ ]:
# --- Unzip Mendeley V4 ---
print('=== Mendeley V4 Dataset ===')
print(f'Zip exists: {os.path.exists(MENDELEY_ZIP)}')

# List contents without extracting
with zipfile.ZipFile(MENDELEY_ZIP, 'r') as zf:
    print(f'\nFiles in zip ({len(zf.namelist())} total):')
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f'  {name}  ({info.file_size:,} bytes)')

# Extract to a temp location
extract_dir = os.path.join(DRIVE_BASE, '_mendeley_extracted')
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(MENDELEY_ZIP, 'r') as zf:
    zf.extractall(extract_dir)
print(f'\n✅ Extracted to: {extract_dir}')

# Walk to find CSV files
csv_files = []
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))
            
print(f'\nFound {len(csv_files)} CSV file(s):')
for cf in csv_files:
    print(f'  {cf}')

In [ ]:
# --- Examine each CSV ---
mendeley_dfs = []

for cf in csv_files:
    print(f'\n{"="*60}')
    print(f'File: {os.path.basename(cf)}')
    
    # Try common encodings
    for enc in ['utf-8', 'latin-1', 'cp1256']:
        try:
            df = pd.read_csv(cf, encoding=enc)
            break
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    else:
        print('  ⚠️ Could not read this file')
        continue
    
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Encoding used: {enc}')
    print(f'\n  First 3 rows:')
    print(df.head(3).to_string())
    
    # Show label distribution if label column exists
    label_cols = [c for c in df.columns if any(k in c.lower() for k in ['label', 'class', 'target', 'offensive', 'toxic'])]
    if label_cols:
        for lc in label_cols:
            print(f'\n  Label distribution ({lc}):')
            print(df[lc].value_counts().to_string())
    
    mendeley_dfs.append((cf, df))

## 3. Explore Dataset 2: GitHub OMCD (kabilessefar)

In [ ]:
# --- Explore GitHub OMCD ---
print('=== GitHub OMCD Dataset ===')

# List directory structure
for root, dirs, files in os.walk(GITHUB_DIR):
    level = root.replace(GITHUB_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath)
        print(f'{subindent}{f}  ({fsize:,} bytes)')

In [ ]:
# --- Load and examine train + test ---
github_dfs = {}

for split_name in ['train', 'test']:
    # Try both possible paths
    possible_paths = [
        os.path.join(GITHUB_DIR, f'{split_name}_OMCD', f'{split_name}.csv'),
        os.path.join(GITHUB_DIR, f'{split_name}.csv'),
    ]
    
    for p in possible_paths:
        if os.path.exists(p):
            for enc in ['utf-8', 'latin-1', 'cp1256']:
                try:
                    df = pd.read_csv(p, encoding=enc)
                    github_dfs[split_name] = df
                    print(f'\n{"="*60}')
                    print(f'{split_name}.csv loaded from: {p}')
                    print(f'  Shape: {df.shape}')
                    print(f'  Columns: {list(df.columns)}')
                    print(f'\n  First 5 rows:')
                    print(df.head(5).to_string())
                    
                    label_cols = [c for c in df.columns if any(k in c.lower() for k in ['label', 'class', 'target', 'offensive', 'toxic'])]
                    if label_cols:
                        for lc in label_cols:
                            print(f'\n  Label distribution ({lc}):')
                            print(df[lc].value_counts().to_string())
                    break
                except (UnicodeDecodeError, pd.errors.ParserError):
                    continue
            break
    else:
        print(f'⚠️ {split_name}.csv not found in expected paths')

## 4. Unify & Merge Strategy

**Goal**: Create a single DataFrame with columns:
- `text`: The sentence/comment text
- `label`: Binary label (0 = non-offensive, 1 = offensive)
- `source`: Which dataset it came from (`mendeley_v4` or `github_omcd`)
- `original_label`: The original label from the source dataset (for traceability)

**Deduplication**: Exact text match + normalized text match (lowercase, stripped whitespace).

In [ ]:
# =============================================================
# ADAPTIVE MERGE — Adjust column names based on what we found
# =============================================================
# Run this cell AFTER examining the outputs above.
# It will auto-detect the text and label columns.

def find_text_col(df):
    """Auto-detect the text/sentence column."""
    for candidate in ['Sentence', 'sentence', 'text', 'Text', 'comment', 'Comment', 'tweet', 'Tweet', 'content']:
        if candidate in df.columns:
            return candidate
    # Fallback: first string column
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].str.len().mean() > 20:
            return col
    return None

def find_label_col(df):
    """Auto-detect the label column."""
    for candidate in ['Label', 'label', 'offensive', 'Offensive', 'is_offensive', 'toxicity', 'class']:
        if candidate in df.columns:
            return candidate
    # Fallback: first numeric column with 2-5 unique values
    for col in df.columns:
        if df[col].nunique() <= 10 and df[col].dtype in ['int64', 'float64']:
            return col
    return None


unified_parts = []

# --- Process Mendeley V4 ---
print('=== Processing Mendeley V4 ===')
for cf, df in mendeley_dfs:
    text_col = find_text_col(df)
    label_col = find_label_col(df)
    print(f'  File: {os.path.basename(cf)} → text_col="{text_col}", label_col="{label_col}"')
    
    if text_col is None or label_col is None:
        print(f'  ⚠️ Skipping — could not detect columns')
        continue
    
    sub = pd.DataFrame({
        'text': df[text_col].astype(str).str.strip(),
        'label': df[label_col],
        'source': 'mendeley_v4',
        'original_label': df[label_col].astype(str),
    })
    
    # Normalize labels to binary (0/1)
    unique_labels = sub['label'].unique()
    print(f'  Unique labels: {unique_labels}')
    
    # If labels are strings, map them
    if sub['label'].dtype == 'object':
        # Common patterns: 'offensive'/'non-offensive', 'OFF'/'NOT', etc.
        label_map = {}
        for lbl in unique_labels:
            lbl_lower = str(lbl).lower().strip()
            if any(k in lbl_lower for k in ['offensive', 'toxic', 'hate', 'harmful', '1']):
                label_map[lbl] = 1
            else:
                label_map[lbl] = 0
        sub['label'] = sub['label'].map(label_map)
        print(f'  Label mapping: {label_map}')
    
    # Drop rows with empty text
    sub = sub[sub['text'].str.len() > 2].copy()
    print(f'  Rows after cleaning: {len(sub)}')
    unified_parts.append(sub)

# --- Process GitHub OMCD ---
print('\n=== Processing GitHub OMCD ===')
for split_name, df in github_dfs.items():
    text_col = find_text_col(df)
    label_col = find_label_col(df)
    print(f'  Split: {split_name} → text_col="{text_col}", label_col="{label_col}"')
    
    if text_col is None or label_col is None:
        print(f'  ⚠️ Skipping — could not detect columns')
        continue
    
    sub = pd.DataFrame({
        'text': df[text_col].astype(str).str.strip(),
        'label': df[label_col],
        'source': f'github_omcd_{split_name}',
        'original_label': df[label_col].astype(str),
    })
    
    # Normalize labels
    unique_labels = sub['label'].unique()
    print(f'  Unique labels: {unique_labels}')
    
    if sub['label'].dtype == 'object':
        label_map = {}
        for lbl in unique_labels:
            lbl_lower = str(lbl).lower().strip()
            if any(k in lbl_lower for k in ['offensive', 'toxic', 'hate', 'harmful', '1']):
                label_map[lbl] = 1
            else:
                label_map[lbl] = 0
        sub['label'] = sub['label'].map(label_map)
        print(f'  Label mapping: {label_map}')
    
    sub = sub[sub['text'].str.len() > 2].copy()
    print(f'  Rows after cleaning: {len(sub)}')
    unified_parts.append(sub)

print(f'\n=== Total parts to merge: {len(unified_parts)} ===')
for i, p in enumerate(unified_parts):
    print(f'  Part {i}: {len(p)} rows, source={p["source"].iloc[0]}')

In [ ]:
# =============================================================
# MERGE + DEDUPLICATE
# =============================================================
if not unified_parts:
    raise ValueError('No data parts found! Check the exploration cells above.')

merged = pd.concat(unified_parts, ignore_index=True)
print(f'Total rows BEFORE dedup: {len(merged):,}')

# Ensure label is integer
merged['label'] = merged['label'].astype(int)

# Dedup by exact text
merged = merged.drop_duplicates(subset=['text'], keep='first')
print(f'After exact text dedup: {len(merged):,}')

# Dedup by normalized text (lowercase, strip extra whitespace)
merged['_norm_key'] = merged['text'].str.lower().str.strip()
merged = merged.drop_duplicates(subset=['_norm_key'], keep='first')
merged = merged.drop(columns=['_norm_key'])
print(f'After normalized dedup: {len(merged):,}')

# Remove empty or very short texts
merged = merged[merged['text'].str.len() >= 3].copy()
print(f'After removing short texts: {len(merged):,}')

# Show final distribution
print(f'\n=== Final Dataset Statistics ===')
print(f'Total samples: {len(merged):,}')
print(f'\nLabel distribution:')
print(merged['label'].value_counts().to_string())
print(f'\nOffensive ratio: {merged["label"].mean():.3f}')
print(f'\nSource breakdown:')
print(merged['source'].value_counts().to_string())

# Sample from each source
print(f'\n=== Sample Rows ===')
for src in merged['source'].unique():
    print(f'\n--- Source: {src} ---')
    sample = merged[merged['source'] == src].sample(min(3, len(merged[merged['source'] == src])))
    for _, row in sample.iterrows():
        print(f'  [{row["label"]}] {row["text"][:100]}')

## 5. Train / Val / Test Split

**Strategy**:
- Keep the GitHub OMCD `test.csv` as a **held-out test set** (it was already separated by the original authors)
- Split the remaining data 85/15 into train/val
- This gives us a fair comparison: Mendeley V4 + GitHub train in training, GitHub test for evaluation

In [ ]:
from sklearn.model_selection import train_test_split

# Separate GitHub test set as held-out evaluation
heldout_test = merged[merged['source'] == 'github_omcd_test'].copy()
remaining = merged[merged['source'] != 'github_omcd_test'].copy()

print(f'Held-out test set (GitHub OMCD test): {len(heldout_test):,}')
print(f'Remaining for train/val split: {len(remaining):,}')

# Stratified split of remaining data: 85% train, 15% val
train_df, val_df = train_test_split(
    remaining,
    test_size=0.15,
    random_state=42,
    stratify=remaining['label']
)

print(f'\n=== Final Splits ===')
print(f'Train: {len(train_df):,}  (offensive: {train_df["label"].sum():,}, ratio: {train_df["label"].mean():.3f})')
print(f'Val:   {len(val_df):,}  (offensive: {val_df["label"].sum():,}, ratio: {val_df["label"].mean():.3f})')
print(f'Test:  {len(heldout_test):,}  (offensive: {heldout_test["label"].sum():,}, ratio: {heldout_test["label"].mean():.3f})')

# Also create a combined test set (heldout + 10% of remaining) for robust evaluation
test_extra, train_df_final = train_test_split(
    train_df,
    test_size=0.90,
    random_state=42,
    stratify=train_df['label']
)
combined_test = pd.concat([heldout_test, test_extra], ignore_index=True)

print(f'\n=== Alternative: Combined Test Set ===')
print(f'Combined test: {len(combined_test):,}  (offensive: {combined_test["label"].sum():,}, ratio: {combined_test["label"].mean():.3f})')
print(f'Train (reduced): {len(train_df_final):,}')

In [ ]:
# =============================================================
# SAVE ALL SPLITS
# =============================================================

# Primary splits
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train_merged.csv'), index=False, encoding='utf-8')
val_df.to_csv(os.path.join(OUTPUT_DIR, 'val_merged.csv'), index=False, encoding='utf-8')
heldout_test.to_csv(os.path.join(OUTPUT_DIR, 'test_heldout.csv'), index=False, encoding='utf-8')

# Full merged (for reference)
merged.to_csv(os.path.join(OUTPUT_DIR, 'full_merged.csv'), index=False, encoding='utf-8')

# Also save the combined test set variant
train_df_final.to_csv(os.path.join(OUTPUT_DIR, 'train_merged_v2.csv'), index=False, encoding='utf-8')
combined_test.to_csv(os.path.join(OUTPUT_DIR, 'test_combined.csv'), index=False, encoding='utf-8')

print(f'✅ All files saved to: {OUTPUT_DIR}')
print(f'\nFiles created:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    fsize = os.path.getsize(fpath)
    print(f'  {f}  ({fsize:,} bytes)')

## 6. Quick Stats Summary (for PFE Report)

Run this after saving to get a summary table for your report.

In [ ]:
print('╔══════════════════════════════════════════════════╗')
print('║   🇲🇦 AEGIS Darija Dataset — Merge Summary       ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Mendeley V4 samples:    {len(pd.concat([p for _, p in mendeley_dfs])):>8,}  ║')
print(f'║  GitHub OMCD samples:   {sum(len(d) for d in github_dfs.values()):>8,}  ║')
print(f'║  After dedup merge:     {len(merged):>8,}  ║')
print(f'║  Offensive ratio:       {merged["label"].mean():>8.3f}  ║')
print(f'║  Train set:             {len(train_df):>8,}  ║')
print(f'║  Validation set:        {len(val_df):>8,}  ║')
print(f'║  Test (held-out):       {len(heldout_test):>8,}  ║')
print('╚══════════════════════════════════════════════════╝')